# Лабораторна робота № 8 
**Тема: Структура даних граф. Алгоритми на графах** 
**Мета: Засвоїти представлення структури даних граф та основні алгоритми роботи з ними засобами Python.**
**Виконав Голубенко Денис**

---

## Короткі теоретичні відомості 

**Граф** — це абстрактний тип даних, який складається з множини вершин (вузлів) $V$ та множини ребер (дуг) $E$, які описують зв'язки між ними. Математично граф задається як $G = (V, E)$.

### Основні поняття:
* **Вершина (вузол):** Головний елемент графа. Може мати унікальний ідентифікатор («ключ») та додаткові дані («корисне навантаження»).
* **Ребро (дуга):** Зв'язок між двома вершинами. Ребра бувають:
  * *Односпрямовані (орієнтовані):* рух дозволено тільки в один бік. Граф із такими ребрами називається **диграфом**.
  * *Двоспрямовані (неорієнтовані):* рух в обидва боки.
* **Вага ребра:** Числове значення (вартість, відстань, час), необхідне для переходу від однієї вершини до іншої.
* **Шлях:** Послідовність вершин, з'єднаних ребрами.
* **Цикл:** Шлях, який починається і завершується в одній і тій самій вершині.

## Способи представлення графа в комп'ютері

1. **Матриця суміжності:** Двовимірний масив розміром $|V| \times |V|$. Зручна для щільних графів, де ребер дуже багато.
2. **Список суміжності:** Словник або масив списків, де для кожної вершини зберігаються тільки наявні сусіди та ваги. Використовується для розріджених графів для економії пам'яті.

### Порівняння асимптотичної складності операцій:

| Операція | Список суміжності | Матриця суміжності |
| :--- | :---: | :---: |
| **Перевірка на наявність ребра (x, y)** | $O(|V|)$ | $O(1)$ |
| **Визначення степені вершини** | $O(1)$ | $O(|V|)$ |
| **Використання пам'яті** | $O(|V| + |E|)$ | $O(|V|^2)$ |
| **Вставляння/видалення грані** | $O(1)$ | $O(1)$ |
| **Обхід графа** | $O(|V| + |E|)$ | $O(|V|^2)$ |

In [1]:
import networkx as nx
import matplotlib.pyplot as plt

print("--- Створення орієнтованого зваженого графа ---")
G_directed = nx.DiGraph()

edges_weights = [
    ('V0', 'V1', 5), ('V0', 'V5', 2),
    ('V1', 'V2', 4),
    ('V2', 'V3', 9),
    ('V3', 'V4', 7), ('V3', 'V5', 3),
    ('V4', 'V0', 1),
    ('V5', 'V2', 1), ('V5', 'V4', 8)
]
G_directed.add_weighted_edges_from(edges_weights)

plt.figure(figsize=(7, 5))
pos_dir = nx.spring_layout(G_directed, seed=42)
nx.draw_networkx_nodes(G_directed, pos_dir, node_size=700, node_color='lightgreen')
nx.draw_networkx_labels(G_directed, pos_dir, font_size=12, font_family='sans-serif')
nx.draw_networkx_edges(G_directed, pos_dir, arrowstyle='->', arrowsize=18, width=2, edge_color='gray')
edge_labels_dir = nx.get_edge_attributes(G_directed, 'weight')
nx.draw_networkx_edge_labels(G_directed, pos_dir, edge_labels=edge_labels_dir, font_size=10)
plt.title("Орієнтований зважений граф (Рис. 1.6)")
plt.axis('off')
plt.show()

print("\n--- Створення неорієнтованого графа для DFS/BFS ---")
G_undirected = nx.Graph()
G_undirected.add_nodes_from(['A', 'B', 'C', 'D', 'E', 'F'])
G_undirected.add_edges_from([('A', 'B'), ('A', 'C'), ('B', 'D'), ('B', 'E'), ('C', 'F'), ('E', 'F')])

plt.figure(figsize=(7, 5))
pos_undir = nx.spring_layout(G_undirected, seed=10)
nx.draw_networkx_nodes(G_undirected, pos_undir, node_size=700, node_color='skyblue')
nx.draw_networkx_labels(G_undirected, pos_undir, font_size=14, font_family='sans-serif')
nx.draw_networkx_edges(G_undirected, pos_undir, width=2, edge_color='blue')
plt.title("Неорієнтований граф для пошуку в глибину та ширину")
plt.axis('off')
plt.show()

ModuleNotFoundError: No module named 'networkx'

In [ ]:
import networkx as nx

print("==================================================")
print("РЕЗУЛЬТАТИ РОБОТИ АЛГОРИТМІВ НАЙКОРОТШИХ ШЛЯХІВ")
print("==================================================")

print("\n[Алгоритм Дейкстри]")
lengths_dijkstra, paths_dijkstra = nx.single_source_dijkstra(G_directed, source='V0')
print(f"Найкоротша відстань від V0 до V4: {lengths_dijkstra['V4']}")
print(f"Шлях від V0 до V4: {paths_dijkstra['V4']}")
print(f"Всі найкоротші відстані від вузла V0: {lengths_dijkstra}")

print("\n[Алгоритм Беллмана-Форда]")
lengths_bf = nx.single_source_bellman_ford_path_length(G_directed, source='V0')
print(f"Найкоротша відстань від V0 до V4: {lengths_bf['V4']}")
print(f"Всі найкоротші відстані від вузла V0: {lengths_bf}")

In [ ]:
import networkx as nx

def dfs(graph, start, visited=None):
    if visited is None:
        visited = set()
    visited.add(start)
    for next_node in set(graph.adj[start].keys()) - visited:
        dfs(graph, next_node, visited)
    return visited

def dfs_paths(graph, start, goal, path=None):
    if path is None:
        path = [start]
    if start == goal:
        yield path
    for next_node in set(graph.adj[start].keys()) - set(path):
        yield from dfs_paths(graph, next_node, goal, path + [next_node])

def bfs_paths(graph, start, goal):
    queue = [(start, [start])]
    while queue:
        (vertex, path) = queue.pop(0)
        for next_node in set(graph.adj[vertex].keys()) - set(path):
            if next_node == goal:
                yield path + [next_node]
            else:
                queue.append((next_node, path + [next_node]))

print("==================================================")
print("РЕЗУЛЬТАТИ ОБХОДІВ ГРАФА (DFS та BFS)")
print("==================================================")

print("\n[Пошук у глибину - DFS]")
print(f"Повний обхід DFS починаючи з вершини 'C': {dfs(G_undirected, 'C')}")
print(f"Всі можливі шляхи від 'C' до 'F' (через DFS): {list(dfs_paths(G_undirected, 'C', 'F'))}")

print("\n[Пошук у ширину - BFS]")
print(f"Всі шляхи від 'A' до 'F' (через BFS): {list(bfs_paths(G_undirected, 'A', 'F'))}")

## Відповіді на контрольні питання
### 1. Що таке граф у термінах теорії графів? Наведіть приклади реальних ситуацій, де можна застосовувати графи.
Граф — це сукупність вершин та ребер, які зв'язують ці вершини між собою. 
* *Приклади застосування:* Моделювання комп'ютерних мереж (маршрутизатори), соціальних зв'язків (знайомства користувачів), логістики (маршрути вантажоперевезень).
### 2. Які основні види графів існують? Наведіть відмінності між орієнтованими і неорієнтованими графами.
Графи бувають орієнтовані, неорієнтовані, зважені, незважені, ациклічні (DAG).
* *Відмінність:* У неорієнтованому графі зв'язок діє в обидва боки рівноцінно. В орієнтованому графі ребро має напрямок, тобто рух дозволено лише від початкової вершини до кінцевої.
### 3. Як можна представити граф у пам’яті комп'ютера? Опишіть структури даних, які використовуються для зберігання графів.
Граф представляється за допомогою **матриці суміжності** (двовимірна таблиця суміжності зв'язків) або **списку суміжності** (словник, де ключ — вершина, а значення — список її суміжних сусідів).
### 4. Як працює алгоритм пошуку в ширину (BFS) на графах? Наведіть приклади ситуацій, де застосовується цей алгоритм.
BFS покроково перебирає всі сусідні вершини поточного рівня ("в ширину"), перш ніж перейти на наступний крок заглиблення. Для роботи використовує чергу (FIFO).
* *Застосування:* Знаходження найкоротших шляхів за кількістю ребер у незважених графах.
### 5. Що таке алгоритм пошуку в глибину (DFS) на графах? Як він відрізняється від BFS? Дайте приклади задач, де використовується DFS.
DFS йде по одній гілці графа до самого кінця ("в глибину"), а коли заходить у тупик, повертається назад (backtracking). Використовує стек або рекурсію. На відміну від BFS, він не гарантує знаходження найкоротшого шляху першим.
* *Застосування:* Визначення наявності циклів, топологічне сортування.
### 6. Опишіть алгоритм Дейкстри для пошуку найкоротшого шляху в графі. Які умови повинні виконуватися для успішної роботи цього алгоритму?
Алгоритм Дейкстри є жадібним алгоритмом, який шукає найкоротші відстані від однієї обраної вершини до всіх інших. Його головна умова — **вага ребер графа не повинна бути від'ємною**.

---

## Висновок
Під час виконання лабораторної роботи № 8 було детально вивчено структуру даних "граф". За допомогою мови програмування Python та бібліотеки `NetworkX` було успішно реалізовано побудову орієнтованих і неорієнтованих зважених моделей графів, виконано їх графічну візуалізацію, а також досліджено роботу базових алгоритмів обходу (DFS, BFS) та алгоритмів пошуку найкоротших шляхів (Дейкстри, Беллмана-Форда).